# PyTorch DataLoaders for RFInject

This notebook builds geographic train, validation, and test burst-level PyTorch `Dataset` and `DataLoader` objects on top of the RFInject workflow introduced in [how_to_start.ipynb](./how_to_start.ipynb).

## Why burst-level loading matters

In raw SAR products, a scene is naturally split into bursts. RFInject keeps that structure in Zarr, which means each burst can be inspected, sampled, mirrored, and batched independently. That is useful when scenes are large, when burst shapes vary, and when you want train/validation/test splits without flattening the original product hierarchy.

## What this notebook does

- keeps the data access pattern bucket-native
- mirrors metadata first so scene structure can be inspected cheaply
- reads `src/RFInject-full.csv` and uses the `Split` column as the source of truth for train, validation, and test assignment
- intersects catalog child products with the scenes currently available in the Hugging Face bucket
- saves the selected train, validation, and test subset under `./data` before the DataLoaders iterate
- exposes a `prefetch_selected_bursts` flag to mirror only the selected bursts for each split
- exposes a `download_full_scene` flag to cache each selected scene before iteration
- exposes a `sample_fraction` flag to target a fraction of each scene payload, for example `0.1` for 10%
- builds separate train, validation, and test loaders

## Assumptions and constraints

- the unit of sampling is a full burst from one RFInject Zarr scene
- `sample_fraction` is applied independently to each scene in deterministic burst order
- bursts are atomic download units, so the realized fraction can be slightly above the requested one
- with `prefetch_selected_bursts=True`, only the selected bursts are mirrored under `./data` and the DataLoader reads from that local copy
- set `allow_remote_fetch=True` only if you want the older lazy-download behavior on first access
- batching uses padding so the dataloader still works if bursts have different spatial shapes
- only child scenes currently present in the Hugging Face bucket can be loaded


## Environment setup

This notebook needs the project dependencies, the Jupyter extras, and PyTorch in the same environment as the notebook kernel.

### PDM

```bash
pdm install -G jupyter_env
pdm run python -m pip install torch
```

### virtualenv

```bash
python3 -m pip install --user virtualenv
python3 -m virtualenv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -e ".[jupyter_env]"
python -m pip install torch
```

### uv

```bash
uv sync --extra jupyter_env
uv pip install --python .venv/bin/python torch
```

If you want to run multi-worker loading later, make sure the same environment is available to every worker process.


## Autoreload

This keeps local `rfinject` edits visible without restarting the kernel while you iterate on dataset and dataloader code.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import zarr

try:
    import torch
    from torch.utils.data import DataLoader
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'PyTorch is required for this notebook. Install it with `python -m pip install torch` and re-run the notebook.'
    ) from exc

from rfinject import (
    DEFAULT_HF_BUCKET_ID,
    list_hf_bucket_zarrs,
    open_hf_bucket_zarr,
)
from rfinject.pytorch_data import (
    RFInjectSplitBurstDataset,
    burst_sort_key,
    describe_access_mode,
    pad_burst_batch,
    select_bursts_by_fraction,
    tensor_magnitude,
)

print('Python version:', sys.version)
print('Zarr version:', zarr.__version__)
print('Torch version:', torch.__version__)

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'rfinject').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


repo_root = find_repo_root(Path.cwd())
catalog_path = repo_root / 'src/RFInject-full.csv'
cache_dir = repo_root / 'data'
cache_dir.mkdir(parents=True, exist_ok=True)
artifact_dir = repo_root / 'outputs/pytorch_dataloader_artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

# Set this to True to cache every file in each selected scene before building the DataLoaders.
download_full_scene = False
prefetch_selected_bursts = True
allow_remote_fetch = False

# Target a fraction of each scene payload. Bursts stay atomic, so the realized fraction can be slightly higher.
sample_fraction = 0.2

batch_size = 2
num_workers = 0
train_shuffle = True
eval_shuffle = False
train_seed = 7
rfi_channel = 0
preview_split = 'train'
max_scenes_per_split = None

if not 0 < sample_fraction <= 1:
    raise ValueError('sample_fraction must be in the interval (0, 1].')
if preview_split not in {'train', 'validation', 'test'}:
    raise ValueError("preview_split must be one of 'train', 'validation', or 'test'.")
if max_scenes_per_split is not None and max_scenes_per_split < 1:
    raise ValueError('max_scenes_per_split must be None or a positive integer.')

print('Repo root:', repo_root)
print('Catalog path:', catalog_path)
print('Data root:', cache_dir)


In [ ]:
def parse_child_products(raw_value: str) -> list[str]:
    if pd.isna(raw_value) or raw_value == '':
        return []
    return [f"{child}.zarr" for child in json.loads(raw_value)]


catalog_df = pd.read_csv(catalog_path)
split_column = 'Split'
geo_section_column = 'GeoSection'
required_columns = {'Name', 'ChildProducts', geo_section_column, split_column}
missing_columns = required_columns.difference(catalog_df.columns)
if missing_columns:
    raise KeyError(f'Missing required catalog columns: {sorted(missing_columns)}')

bucket_scene_paths = set(list_hf_bucket_zarrs(DEFAULT_HF_BUCKET_ID))
expected_splits = ('train', 'validation', 'test')
split_scene_entries = {split_name: [] for split_name in expected_splits}
catalog_child_counts = {split_name: 0 for split_name in expected_splits}

for split_name in expected_splits:
    split_rows = catalog_df.loc[catalog_df[split_column] == split_name]
    for row in split_rows.itertuples(index=False):
        child_scene_paths = parse_child_products(row.ChildProducts)
        catalog_child_counts[split_name] += len(child_scene_paths)
        for scene_path in child_scene_paths:
            if scene_path in bucket_scene_paths:
                split_scene_entries[split_name].append(
                    {
                        'scene_path': scene_path,
                        'geo_section': getattr(row, geo_section_column),
                        'parent_name': row.Name,
                    }
                )

for split_name, scene_entries in split_scene_entries.items():
    unique_scene_entries = {}
    for entry in scene_entries:
        unique_scene_entries[entry['scene_path']] = entry
    split_scene_entries[split_name] = [
        unique_scene_entries[scene_path]
        for scene_path in sorted(unique_scene_entries)
    ]
    if max_scenes_per_split is not None:
        split_scene_entries[split_name] = split_scene_entries[split_name][:max_scenes_per_split]

available_scene_counts = {
    split_name: len(scene_entries)
    for split_name, scene_entries in split_scene_entries.items()
}
available_section_counts = {
    split_name: len({entry['geo_section'] for entry in scene_entries})
    for split_name, scene_entries in split_scene_entries.items()
}
empty_splits = [split_name for split_name, count in available_scene_counts.items() if count == 0]
if empty_splits:
    raise RuntimeError(f'No bucket-available scenes were found for split(s): {empty_splits}')

print('Catalog parents:', len(catalog_df))
print(f"Split source column: {split_column}")
print('Bucket scenes available:', len(bucket_scene_paths))
for split_name in expected_splits:
    print(
        f"{split_name.title()}: {available_scene_counts[split_name]} available scenes from "
        f"{catalog_child_counts[split_name]} catalog child products across "
        f"{available_section_counts[split_name]} geographic sections"
    )
    print('  First scenes:', [entry['scene_path'] for entry in split_scene_entries[split_name][:3]])

preview_scene_path = split_scene_entries[preview_split][0]['scene_path']

metadata_group = open_hf_bucket_zarr(
    DEFAULT_HF_BUCKET_ID,
    preview_scene_path,
    local_dir=cache_dir,
    metadata_only=True,
)

all_bursts = sorted(
    (name for name in metadata_group.keys() if name.startswith('burst_')),
    key=burst_sort_key,
)
if not all_bursts:
    raise RuntimeError(f'No bursts were found in {preview_scene_path}.')

burst_sizes, total_product_bytes, target_bytes, selected_bursts, selected_bytes, actual_fraction = select_bursts_by_fraction(
    metadata_group,
    all_bursts,
    sample_fraction,
)

echo_shapes = {burst_name: tuple(metadata_group[burst_name]['echo'].shape) for burst_name in all_bursts}
rfi_shapes = {burst_name: tuple(metadata_group[burst_name]['rfi'].shape) for burst_name in all_bursts}

print()
print(f'Preview split: {preview_split}')
print(f'Preview scene: {preview_scene_path}')
print(f'Total bursts available: {len(all_bursts)}')
print(f'Total payload volume: {total_product_bytes / (1024 ** 3):.2f} GiB')
print(f'Requested payload fraction: {sample_fraction:.1%} ({target_bytes / (1024 ** 3):.2f} GiB)')
print(f'Selected bursts: {len(selected_bursts)}')
print(f'Actual selected payload: {selected_bytes / (1024 ** 3):.2f} GiB ({actual_fraction:.1%} of product)')
print('Selected bursts:', selected_bursts)
print('Unique echo shapes:', sorted(set(echo_shapes.values())))
print('Unique rfi shapes:', sorted(set(rfi_shapes.values())))


In [ ]:
# Batch collation and visualization helpers are imported from rfinject.pytorch_data.


In [ ]:
# RFInjectSplitBurstDataset is imported from rfinject.pytorch_data.


In [ ]:
datasets = {
    split_name: RFInjectSplitBurstDataset(
        split_scene_entries[split_name],
        split_name=split_name,
        cache_dir=cache_dir,
        download_full_scene=download_full_scene,
        prefetch_selected_bursts=prefetch_selected_bursts,
        allow_remote_fetch=allow_remote_fetch,
        sample_fraction=sample_fraction,
        rfi_channel=rfi_channel,
    )
    for split_name in expected_splits
}

train_dataset = datasets['train']
validation_dataset = datasets['validation']
test_dataset = datasets['test']

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=train_shuffle,
    num_workers=num_workers,
    collate_fn=pad_burst_batch,
    generator=torch.Generator().manual_seed(train_seed),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=eval_shuffle,
    num_workers=num_workers,
    collate_fn=pad_burst_batch,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=eval_shuffle,
    num_workers=num_workers,
    collate_fn=pad_burst_batch,
)

loaders = {
    'train': train_loader,
    'validation': validation_loader,
    'test': test_loader,
}

print(
    'Access mode:',
    describe_access_mode(
        download_full_scene=download_full_scene,
        prefetch_selected_bursts=prefetch_selected_bursts,
        allow_remote_fetch=allow_remote_fetch,
    ),
)
for split_name, dataset in datasets.items():
    loader = loaders[split_name]
    selected_fraction = dataset.total_selected_bytes / dataset.total_available_bytes if dataset.total_available_bytes else 0.0
    geo_section_count = len({summary['geo_section'] for summary in dataset.scene_summaries})
    print(
        f"{split_name.title()}: {len(dataset.scene_summaries)} scenes, {len(dataset)} bursts, "
        f"{selected_fraction:.1%} of payload selected across {geo_section_count} sections"
    )
    print(f'  DataLoader batches: {len(loader)}')


In [ ]:
def describe_batch(split_name: str, batch):
    print(f'{split_name.title()} scene paths:', batch['scene_path'])
    print('Geo sections:', batch['geo_section'])
    print('Parent names:', batch['parent_name'])
    print('Burst names:', batch['burst_name'])
    print('Echo batch shape:', tuple(batch['echo'].shape))
    print('RFI batch shape:', tuple(batch['rfi'].shape))
    print('Echo original shapes:', batch['echo_shape'].tolist())
    print('RFI original shapes:', batch['rfi_shape'].tolist())
    print('Valid echo pixels per sample:', batch['echo_mask'].sum(dim=(-2, -1)).tolist())
    print()

first_train_batch = next(iter(train_loader))
first_validation_batch = next(iter(validation_loader))
first_test_batch = next(iter(test_loader))

describe_batch('train', first_train_batch)
describe_batch('validation', first_validation_batch)
describe_batch('test', first_test_batch)

first_batch = first_train_batch


In [ ]:
scene_root = cache_dir / first_train_batch['scene_path'][0]
metadata_files = sorted(path for path in scene_root.rglob('zarr.json'))
chunk_files = sorted(path for path in scene_root.rglob('*') if path.is_file() and path.name != 'zarr.json')

print('Preview training scene root:', scene_root)
print('Metadata files cached:', len(metadata_files))
print('Payload chunk files cached:', len(chunk_files))
print('First cached payload files:')
for path in chunk_files[:10]:
    print(' ', path.relative_to(cache_dir))


In [ ]:
sample_idx = 0
sample_echo = tensor_magnitude(first_train_batch['echo'][sample_idx])
sample_rfi = tensor_magnitude(first_train_batch['rfi'][sample_idx])

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=120)
axes[0].imshow(sample_echo, aspect='auto', cmap='viridis')
axes[0].set_title(f"Train echo magnitude: {first_train_batch['burst_name'][sample_idx]}")
axes[0].set_xlabel('Range')
axes[0].set_ylabel('Azimuth')

axes[1].imshow(sample_rfi, aspect='auto', cmap='magma')
axes[1].set_title(f"Train RFI channel {rfi_channel}: {first_train_batch['burst_name'][sample_idx]}")
axes[1].set_xlabel('Range')
axes[1].set_ylabel('Azimuth')

fig.tight_layout()
preview_path = artifact_dir / 'train_dataloader_batch_preview.png'
fig.savefig(preview_path, bbox_inches='tight')
plt.show()
print('Preview scene path:', first_train_batch['scene_path'][sample_idx])
print('Saved preview to', preview_path)


## Practical notes

- A product is one top-level `.zarr` store, and a burst is the atomic sample unit used by this notebook.
- This notebook uses the `Split` column in `src/RFInject-full.csv` as the only source of truth for train, validation, and test membership.
- `GeoSection` is carried through as metadata for inspection and debugging, but it is not used to recompute the split inside the notebook.
- Only child scenes that are both listed in `ChildProducts` and currently present in the Hugging Face bucket are included in the loaders.
- `metadata_only=True` is used first so the scene structure can be inspected before chunk payloads are mirrored.
- Set `max_scenes_per_split` to a small integer when you want a faster smoke run without changing the geographic split logic.
- `train_loader` shuffles deterministically with `train_seed`; `validation_loader` and `test_loader` keep stable order.
- The notebook writes its local mirror under `./data` by default.
- Keep `prefetch_selected_bursts = True` to mirror only the selected bursts for each split before iteration.
- Set `download_full_scene = True` if you want every selected scene mirrored before the first batch is read.
- Set `allow_remote_fetch = True` only if you want lazy burst downloads on first access instead of strict local-only loading.
- Increase or decrease `sample_fraction` to target a fraction of each scene payload rather than a fraction of the split burst count.
- Because bursts are atomic, the realized mirrored payload can be slightly above the requested fraction.
- The strict geographic split is not recomputed here; the notebook consumes the split already written into the catalog.
